# Analisis Duplikat v2

Tujuan: menguji asal duplikat, dampak penghapusannya pada populasi, dan dampaknya pada evaluasi serta pelatihan.
Notebook ini tidak mengubah `dataset/` maupun `EDA_Project_Churn.ipynb`.

Hasilnya berupa bukti, bukan pembuktian asal data, karena dataset tidak punya ID pelanggan.

**Aturan keputusan (ditetapkan sebelum melihat hasil)**
- Opsi 1 (hapus duplikat persis) tetap dipakai jika pola salinan kuat (A) dan selisih pada langkah 7 mencakup nol.
- Jika bias penghapusan besar (B: SMD > 0,2 atau classifier test jelas di atas 0,5), hal itu ditulis sebagai keterbatasan dan opsi 1 dibahas ulang.
- Jika pola salinan tidak terbukti (A), opsi 1 dibandingkan lagi dengan opsi 2.
- Keputusan akhir ada di tangan pengguna.

## Persiapan

In [1]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

SEED = 180915
N_PERMUTATIONS = 2000

# Path relatif terhadap root project
ROOT = Path.cwd().parent if Path.cwd().name == "program" else Path.cwd()
df = pd.read_csv(ROOT / "dataset" / "Customer Churn.csv")

TARGET = "Churn"
ALL_COLUMNS = df.columns.tolist()
FEATURES = [col for col in ALL_COLUMNS if col != TARGET]
ACTIVITY_COLUMNS = ["Seconds of Use", "Frequency of use", "Frequency of SMS"]
is_zero_activity = df[ACTIVITY_COLUMNS].eq(0).all(axis=1).to_numpy()

# Cek angka dasar
deduped = df.drop_duplicates()
assert df.shape == (3150, 14)
assert df.duplicated().sum() == 300
assert len(deduped) == 2850

print(f"Baris: {len(df):,} | setelah dedup: {len(deduped):,}")
print(f"Churn rate: {df[TARGET].mean():.1%} | setelah dedup: {deduped[TARGET].mean():.1%}")
print(f"Bentuk fitur: {df[FEATURES].shape}")
print(f"Baris beraktivitas nol: {is_zero_activity.sum()}")

Baris: 3,150 | setelah dedup: 2,850
Churn rate: 15.7% | setelah dedup: 15.6%
Bentuk fitur: (3150, 13)
Baris beraktivitas nol: 132


## A1. Uji pola posisi

Jika duplikat muncul secara kebetulan, urutan baris seharusnya tidak berkaitan dengan siapa kembaran siapa.
Statistik uji dihitung ulang pada setiap permutasi agar tidak bergantung pada jarak yang kita lihat lebih dulu:
- `top3_share`: porsi pasangan kembar pada tiga jarak paling sering;
- `longest_run`: blok terpanjang dari baris berurutan yang punya kembaran pada jarak yang sama.

In [2]:
def build_pairs(data):
    # Semua pasangan baris yang identik di seluruh kolom
    groups = data.groupby(ALL_COLUMNS, sort=False).indices.values()
    pairs = [pair for rows in groups if len(rows) > 1 for pair in combinations(sorted(rows), 2)]
    return np.array(pairs)


def top3_share(gaps):
    counts = np.bincount(gaps)
    return np.sort(counts)[-3:].sum() / len(gaps)


def longest_run(starts, gaps):
    # Baris awal berurutan dengan jarak yang sama dihitung sebagai satu blok
    order = np.lexsort((starts, gaps))
    sorted_starts, sorted_gaps = starts[order], gaps[order]
    same_block = (np.diff(sorted_gaps) == 0) & (np.diff(sorted_starts) == 1)
    best = current = 1
    for connected in same_block:
        current = current + 1 if connected else 1
        best = max(best, current)
    return best


pairs = build_pairs(df)
starts, gaps = pairs[:, 0], pairs[:, 1] - pairs[:, 0]
observed_share, observed_run = top3_share(gaps), longest_run(starts, gaps)

rng = np.random.default_rng(SEED)
null_share, null_run = np.empty(N_PERMUTATIONS), np.empty(N_PERMUTATIONS)
for i in range(N_PERMUTATIONS):
    new_position = rng.permutation(len(df))
    a, b = new_position[pairs[:, 0]], new_position[pairs[:, 1]]
    perm_starts, perm_gaps = np.minimum(a, b), np.abs(a - b)
    null_share[i] = top3_share(perm_gaps)
    null_run[i] = longest_run(perm_starts, perm_gaps)

p_share = (1 + (null_share >= observed_share).sum()) / (1 + N_PERMUTATIONS)
p_run = (1 + (null_run >= observed_run).sum()) / (1 + N_PERMUTATIONS)

print(f"Jumlah pasangan kembar: {len(pairs)}")
print(pd.Series(gaps).value_counts().head(8).rename("jumlah_pasangan").to_string())
print()
print(pd.DataFrame({
    "teramati": [observed_share, observed_run],
    "null_rata2": [null_share.mean(), null_run.mean()],
    "null_maks": [null_share.max(), null_run.max()],
    "p_permutasi": [p_share, p_run],
}, index=["top3_share", "longest_run"]).round(4).to_string())

Jumlah pasangan kembar: 457
50      133
950     125
1000    123
300      10
500       8
700       5
200       4
250       4

             teramati  null_rata2  null_maks  p_permutasi
top3_share     0.8337      0.0182     0.0263       0.0005
longest_run   22.0000      1.0405     2.0000       0.0005


In [3]:
# Apakah pola tetap ada pada baris beraktivitas, yang sulit sama secara kebetulan?
pair_is_active = ~is_zero_activity[pairs[:, 0]]
for label, mask in [("aktivitas > 0", pair_is_active), ("aktivitas nol", ~pair_is_active)]:
    print(f"{label}: {mask.sum()} pasangan | top3_share = {top3_share(gaps[mask]):.3f} | "
          f"jarak teratas = {pd.Series(gaps[mask]).value_counts().head(3).to_dict()}")

# Lokasi blok untuk tiga jarak paling sering
print()
for gap in pd.Series(gaps).value_counts().head(3).index:
    positions = np.sort(starts[gaps == gap])
    blocks = np.split(positions, np.where(np.diff(positions) != 1)[0] + 1)
    summary = sorted(((blk[0], blk[-1], len(blk)) for blk in blocks), key=lambda x: -x[2])[:5]
    print(f"Jarak {gap}: {len(positions)} pasangan, baris awal {positions.min()}-{positions.max()}, "
          f"{len(blocks)} blok, terbesar (awal, akhir, panjang) = {[tuple(map(int, s)) for s in summary]}")

# Sebaran baris yang terlibat duplikasi menurut posisi
involved = df.duplicated(keep=False).to_numpy()
bins = pd.cut(df.index, bins=range(0, 3151, 350), right=False)
print()
print(pd.Series(involved).groupby(bins, observed=True).agg(["sum", "mean"]).round(3)
      .rename(columns={"sum": "baris_terlibat", "mean": "porsi"}).to_string())

aktivitas > 0: 365 pasangan | top3_share = 0.992 | jarak teratas = {50: 130, 1000: 118, 950: 114}
aktivitas nol: 92 pasangan | top3_share = 0.293 | jarak teratas = {950: 11, 300: 10, 500: 6}

Jarak 50: 133 pasangan, baris awal 950-3099, 15 blok, terbesar (awal, akhir, panjang) = [(950, 971, 22), (2000, 2017, 18), (2032, 2049, 18), (3082, 3099, 18), (3050, 3061, 12)]
Jarak 950: 125 pasangan, baris awal 0-2148, 30 blok, terbesar (awal, akhir, panjang) = [(1050, 1067, 18), (2128, 2144, 17), (0, 10, 11), (2100, 2110, 11), (2113, 2121, 9)]
Jarak 1000: 123 pasangan, baris awal 0-2148, 20 blok, terbesar (awal, akhir, panjang) = [(1050, 1071, 22), (2132, 2144, 13), (0, 10, 11), (1077, 1087, 11), (2100, 2110, 11)]

              baris_terlibat  porsi
[0, 350)                  47  0.134
[350, 700)                 6  0.017
[700, 1050)               95  0.271
[1050, 1400)              51  0.146
[1400, 1750)              15  0.043
[1750, 2100)             109  0.311
[2100, 2450)              44  0.

## A2. Jarak ke tetangga terdekat

Fitur distandardisasi hanya untuk mengukur jarak, bukan untuk model.
Jika duplikat adalah salinan, akan ada lonjakan tepat di nol yang terpisah dari sebaran jarak lainnya.
Baris beraktivitas nol dipisahkan karena nilainya lebih mungkin sama secara alami.

In [4]:
scaled = StandardScaler().fit_transform(df[FEATURES])
nearest = NearestNeighbors(n_neighbors=2).fit(scaled)
nn_distance = nearest.kneighbors(scaled)[0][:, 1]
nn_distance = np.where(nn_distance < 1e-9, 0.0, nn_distance)

edges = [-np.inf, 0, 0.05, 0.1, 0.2, 0.5, 1, np.inf]
labels = ["tepat 0", "(0, 0,05]", "(0,05, 0,1]", "(0,1, 0,2]", "(0,2, 0,5]", "(0,5, 1]", "> 1"]
binned = pd.cut(nn_distance, bins=edges, labels=labels)
print(pd.crosstab(binned, np.where(is_zero_activity, "aktivitas_nol", "aktivitas_positif"), margins=True).to_string())

print()
for label, mask in [("aktivitas > 0", ~is_zero_activity), ("aktivitas nol", is_zero_activity)]:
    positive = nn_distance[mask & (nn_distance > 0)]
    print(f"{label}: jarak nol = {(nn_distance[mask] == 0).sum()} | min nonzero = {positive.min():.4f} | "
          f"p1 = {np.percentile(positive, 1):.4f} | median nonzero = {np.median(positive):.4f}")

col_0        aktivitas_nol  aktivitas_positif   All
row_0                                              
tepat 0                 77                399   476
(0, 0,05]                0                 33    33
(0,05, 0,1]              1                 48    49
(0,1, 0,2]              41                466   507
(0,2, 0,5]               9               1649  1658
(0,5, 1]                 2                354   356
> 1                      2                 69    71
All                    132               3018  3150

aktivitas > 0: jarak nol = 399 | min nonzero = 0.0002 | p1 = 0.0471 | median nonzero = 0.2976
aktivitas nol: jarak nol = 77 | min nonzero = 0.0507 | p1 = 0.0863 | median nonzero = 0.1167


## A3. Hampir-duplikat

Hitung jumlah kolom yang berbeda pada setiap pasangan baris.
Jika kesamaan muncul secara alami, pasangan yang berbeda satu atau dua kolom seharusnya jauh lebih banyak daripada yang identik.

In [5]:
values = df[ALL_COLUMNS].to_numpy(dtype=float)
n_rows, n_columns = values.shape
diff_counts = {"semua": np.zeros(n_columns + 1, dtype=np.int64), "aktivitas_positif": np.zeros(n_columns + 1, dtype=np.int64)}
single_column_diff = {"semua": np.zeros(n_columns, dtype=np.int64), "aktivitas_positif": np.zeros(n_columns, dtype=np.int64)}

for i in range(n_rows - 1):
    is_different = values[i + 1:] != values[i]
    n_different = is_different.sum(axis=1)
    both_active = ~is_zero_activity[i] & ~is_zero_activity[i + 1:]
    for key, mask in [("semua", np.ones_like(both_active)), ("aktivitas_positif", both_active)]:
        diff_counts[key] += np.bincount(n_different[mask], minlength=n_columns + 1)
        single_column_diff[key] += is_different[mask & (n_different == 1)].sum(axis=0)

print(pd.DataFrame(diff_counts).rename_axis("kolom_berbeda").head(7).to_string())
print("\nPasangan yang berbeda tepat satu kolom:")
print(pd.DataFrame(single_column_diff, index=ALL_COLUMNS).to_string())

               semua  aktivitas_positif
kolom_berbeda                          
0                457                365
1               1244                 98
2               1609                 75
3               2183                128
4               3859                895
5              13644               9752
6              71558              63264

Pasangan yang berbeda tepat satu kolom:
                         semua  aktivitas_positif
Call  Failure                0                  0
Complains                   60                 33
Subscription  Length      1077                 54
Charge  Amount               1                  0
Seconds of Use               0                  0
Frequency of use             0                  0
Frequency of SMS             0                  0
Distinct Called Numbers      0                  0
Age Group                    0                  0
Tariff Plan                  6                  6
Status                       7                  0

## Ringkasan dampak pada baris

Berapa baris yang akan dihapus menurut aktivitasnya, dan berapa hampir-duplikat yang tetap ada setelah deduplikasi persis.

In [6]:
removed = df.duplicated(keep="first").to_numpy()
print(f"Baris dihapus: {removed.sum()} | aktivitas > 0: {(removed & ~is_zero_activity).sum()} | aktivitas nol: {(removed & is_zero_activity).sum()}")

kept = df.loc[~removed]
kept_values = kept[ALL_COLUMNS].to_numpy(dtype=float)
kept_zero = kept[ACTIVITY_COLUMNS].eq(0).all(axis=1).to_numpy()
remaining = {"semua": np.zeros(n_columns, dtype=np.int64), "aktivitas_positif": np.zeros(n_columns, dtype=np.int64)}
rows_in_near_pair = np.zeros(len(kept), dtype=bool)
for i in range(len(kept) - 1):
    is_different = kept_values[i + 1:] != kept_values[i]
    is_near = is_different.sum(axis=1) == 1
    both_active = ~kept_zero[i] & ~kept_zero[i + 1:]
    remaining["semua"] += is_different[is_near].sum(axis=0)
    remaining["aktivitas_positif"] += is_different[is_near & both_active].sum(axis=0)
    active_near = is_near & both_active
    if active_near.any():
        rows_in_near_pair[i] = True
        rows_in_near_pair[i + 1:][active_near] = True

print("\nPasangan berbeda satu kolom setelah deduplikasi persis:")
print(pd.DataFrame(remaining, index=ALL_COLUMNS).query("semua > 0").to_string())
print(f"\nBaris beraktivitas yang terlibat hampir-duplikat: {rows_in_near_pair.sum()}")
print(f"Bentuk data setelah dedup: {kept.shape} | churn rate: {kept[TARGET].mean():.1%}")

Baris dihapus: 300 | aktivitas > 0: 253 | aktivitas nol: 47

Pasangan berbeda satu kolom setelah deduplikasi persis:
                      semua  aktivitas_positif
Complains                27                 18
Subscription  Length    335                 39
Charge  Amount            1                  0
Tariff Plan               4                  4
Status                    2                  0
Churn                    14                  3

Baris beraktivitas yang terlibat hampir-duplikat: 75
Bentuk data setelah dedup: (2850, 14) | churn rate: 15.6%


## B. Bias akibat penghapusan

Dua definisi diuji: `drop_300` (semua duplikat persis) dan `drop_253` (hanya duplikat persis yang beraktivitas).
Perbandingan memakai satu baris per grup agar salinan tidak menggelembungkan p-value.
SMD adalah selisih rata-rata dibagi simpangan baku gabungan. Batas perhatian mengikuti aturan keputusan: |SMD| > 0,2.

In [7]:
from scipy.stats import chi2_contingency, false_discovery_control, mannwhitneyu
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

removed_all = df.duplicated(keep="first").to_numpy()
scenarios = {
    "keep_all": df,
    "drop_300": df.loc[~removed_all],
    "drop_253": df.loc[~(removed_all & ~is_zero_activity)],
}
assert [len(data) for data in scenarios.values()] == [3150, 2850, 2897]

for name, data in scenarios.items():
    print(f"{name}: baris = {len(data):,} | bentuk fitur = {data[FEATURES].shape} | "
          f"churn = {data[TARGET].sum()} ({data[TARGET].mean():.2%})")

# Satu baris per grup, dengan penanda apakah grup itu punya salinan
group_size = df.groupby(ALL_COLUMNS, sort=False)[ALL_COLUMNS[0]].transform("size").to_numpy()
is_unique_row = ~removed_all
unique_rows = df.loc[is_unique_row].reset_index(drop=True)
has_copies = group_size[is_unique_row] > 1
unique_is_zero = is_zero_activity[is_unique_row]
definitions = {
    "drop_300": has_copies,
    "drop_253": has_copies & ~unique_is_zero,
}
for name, flag in definitions.items():
    print(f"{name}: grup yang punya salinan = {flag.sum()} | grup pembanding = {(~flag).sum()}")

keep_all: baris = 3,150 | bentuk fitur = (3150, 13) | churn = 495 (15.71%)
drop_300: baris = 2,850 | bentuk fitur = (2850, 13) | churn = 446 (15.65%)
drop_253: baris = 2,897 | bentuk fitur = (2897, 13) | churn = 471 (16.26%)
drop_300: grup yang punya salinan = 165 | grup pembanding = 2685
drop_253: grup yang punya salinan = 142 | grup pembanding = 2708


### B1. Perbandingan per grup

In [8]:
BINARY_COLUMNS = ["Complains", "Tariff Plan", "Status", TARGET]


def compare_groups(data, flag):
    rows = []
    for col in ALL_COLUMNS:
        duplicate, other = data.loc[flag, col], data.loc[~flag, col]
        pooled_sd = np.sqrt((duplicate.var(ddof=1) + other.var(ddof=1)) / 2)
        if col in BINARY_COLUMNS:
            p_value = chi2_contingency(pd.crosstab(flag, data[col]))[1]
        else:
            p_value = mannwhitneyu(duplicate, other, alternative="two-sided").pvalue
        rows.append({
            "kolom": col,
            "mean_duplikat": duplicate.mean(),
            "mean_pembanding": other.mean(),
            "smd": (duplicate.mean() - other.mean()) / pooled_sd,
            "p_value": p_value,
        })
    table = pd.DataFrame(rows)
    table["p_fdr"] = false_discovery_control(table["p_value"], method="bh")
    return table.sort_values("smd", key=np.abs, ascending=False)


for name, flag in definitions.items():
    table = compare_groups(unique_rows, flag)
    print(f"\n===== {name} =====")
    print(table.round(3).to_string(index=False))
    print(f"|SMD| > 0,2: {table.loc[table['smd'].abs() > 0.2, 'kolom'].tolist()}")
    print(f"p_fdr < 0,05: {table.loc[table['p_fdr'] < 0.05, 'kolom'].tolist()}")


===== drop_300 =====
                  kolom  mean_duplikat  mean_pembanding    smd  p_value  p_fdr
                 Status          1.339            1.234  0.234    0.003  0.014
         Charge  Amount          0.673            0.993 -0.226    0.006  0.014
Distinct Called Numbers         20.364           24.086 -0.219    0.004  0.014
          Call  Failure          6.424            7.887 -0.207    0.007  0.014
       Frequency of use         60.958           71.070 -0.177    0.005  0.014
         Seconds of Use       3898.188         4573.330 -0.163    0.005  0.014
              Complains          0.055            0.082 -0.110    0.261  0.313
         Customer Value        421.815          478.258 -0.109    0.004  0.014
                  Churn          0.194            0.154  0.105    0.210  0.313
       Frequency of SMS         65.127           74.322 -0.083    0.011  0.019
              Age Group          2.770            2.839 -0.078    0.268  0.313
                    Age       

### B2. Pergeseran dataset

Selisih rata-rata tiap skenario terhadap `keep_all`, dalam satuan simpangan baku `keep_all`.

In [9]:
baseline_mean, baseline_sd = df[ALL_COLUMNS].mean(), df[ALL_COLUMNS].std(ddof=1)
shift = pd.DataFrame({
    name: (scenarios[name][ALL_COLUMNS].mean() - baseline_mean) / baseline_sd
    for name in ["drop_300", "drop_253"]
})
print(shift.round(4).sort_values("drop_300", key=np.abs, ascending=False).to_string())
print(f"\nPergeseran absolut terbesar: {shift.abs().max().round(4).to_dict()}")

summary = pd.DataFrame({
    name: {
        "churn_rate": data[TARGET].mean(),
        "status_nonaktif": data["Status"].eq(2).mean(),
        "complains": data["Complains"].mean(),
        "aktivitas_nol": data[ACTIVITY_COLUMNS].eq(0).all(axis=1).mean(),
    }
    for name, data in scenarios.items()
})
print()
print(summary.round(4).to_string())

                         drop_300  drop_253
Call  Failure              0.0240    0.0066
Charge  Amount             0.0210    0.0106
Distinct Called Numbers    0.0209   -0.0015
Status                    -0.0191    0.0054
Frequency of use           0.0178   -0.0021
Complains                  0.0158    0.0147
Seconds of Use             0.0147   -0.0028
Subscription  Length      -0.0104   -0.0071
Age Group                  0.0101    0.0042
Tariff Plan                0.0096    0.0047
Age                        0.0089    0.0060
Customer Value             0.0078   -0.0071
Frequency of SMS           0.0055   -0.0052
Churn                     -0.0018    0.0149

Pergeseran absolut terbesar: {'drop_300': 0.024, 'drop_253': 0.0149}

                 keep_all  drop_300  drop_253
churn_rate         0.1571    0.1565    0.1626
status_nonaktif    0.2483    0.2400    0.2506
complains          0.0765    0.0807    0.0804
aktivitas_nol      0.0419    0.0298    0.0456


### B3. Classifier two-sample test

Jika model bisa membedakan grup yang punya salinan dari grup lainnya (ROC-AUC jelas di atas 0,5), profil keduanya berbeda.
Hanya memakai 13 fitur, tanpa `Churn` dan tanpa posisi baris.
Label yang diacak dipakai sebagai cek metode, dan hasilnya seharusnya mendekati 0,5.

In [10]:
def two_sample_auc(data, flag):
    model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1)
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
    scores = cross_val_score(model, data[FEATURES], flag, cv=cv, scoring="roc_auc")
    repeat_means = scores.reshape(5, 5).mean(axis=1)
    return {
        "auc_rata2": scores.mean(),
        "fold_min": scores.min(),
        "fold_maks": scores.max(),
        "ulangan_min": repeat_means.min(),
        "ulangan_maks": repeat_means.max(),
    }


rng_b3 = np.random.default_rng(SEED)
results = {}
for name, flag in definitions.items():
    results[name] = two_sample_auc(unique_rows, flag)
    results[f"{name}_label_acak"] = two_sample_auc(unique_rows, rng_b3.permutation(flag))
print(pd.DataFrame(results).T.round(4).to_string())

                     auc_rata2  fold_min  fold_maks  ulangan_min  ulangan_maks
drop_300                0.2912    0.2206     0.3972       0.2842        0.2991
drop_300_label_acak     0.5143    0.4131     0.5924       0.5050        0.5307
drop_253                0.2386    0.1966     0.2922       0.2307        0.2493
drop_253_label_acak     0.5209    0.4634     0.5939       0.4977        0.5441


### B4. Diagnosa AUC di bawah 0,5 pada B3

ROC-AUC RandomForest pada B3 jauh di bawah 0,5, sedangkan label acak mendekati 0,5. Sel ini memeriksa:
- apakah model yang lebih halus (Logistic Regression dan daun lebih besar) memberi hasil berbeda;
- apakah mengelompokkan baris hampir kembar saat CV menghilangkan efeknya;
- komposisi tetangga terdekat tiap kelas.

In [11]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline


def score_summary(scores):
    return f"{np.mean(scores):.4f} ({np.min(scores):.4f}-{np.max(scores):.4f})"


def random_cv():
    return RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)


unique_features = unique_rows[FEATURES].to_numpy(dtype=float)
near_pairs = {k: [] for k in range(4)}
for i in range(len(unique_features) - 1):
    n_different = (unique_features[i + 1:] != unique_features[i]).sum(axis=1)
    for k in near_pairs:
        near_pairs[k].extend((i, i + 1 + j) for j in np.where(n_different == k)[0])
print("Pasangan menurut jumlah fitur berbeda:", {k: len(v) for k, v in near_pairs.items()})


def near_groups(max_diff):
    edges = np.array([pair for k in range(max_diff + 1) for pair in near_pairs[k]])
    graph = coo_matrix((np.ones(len(edges)), (edges[:, 0], edges[:, 1])), shape=(len(unique_rows),) * 2)
    return connected_components(graph, directed=False)[1]


scaled_unique = StandardScaler().fit_transform(unique_rows[FEATURES])
neighbor_index = NearestNeighbors(n_neighbors=11).fit(scaled_unique).kneighbors(scaled_unique)[1][:, 1:]

for name, flag in definitions.items():
    print(f"\n===== {name} (porsi positif {flag.mean():.3f}) =====")
    logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
    print("Logistic Regression:", score_summary(cross_val_score(logistic, unique_rows[FEATURES], flag, cv=random_cv(), scoring="roc_auc")))
    for leaf in [1, 5, 20]:
        forest = RandomForestClassifier(n_estimators=300, class_weight="balanced", min_samples_leaf=leaf, random_state=SEED, n_jobs=-1)
        print(f"RandomForest daun={leaf}:", score_summary(cross_val_score(forest, unique_rows[FEATURES], flag, cv=random_cv(), scoring="roc_auc")))
    for max_diff in [1, 2, 3]:
        scores = []
        for repeat in range(3):
            forest = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1)
            splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + repeat)
            scores.extend(cross_val_score(forest, unique_rows[FEATURES], flag, groups=near_groups(max_diff), cv=splitter, scoring="roc_auc"))
        print(f"RandomForest, grup hampir kembar (beda <= {max_diff} fitur):", score_summary(scores))
    print(f"Porsi positif di 10 tetangga: baris positif = {flag[neighbor_index[flag]].mean():.3f} | "
          f"baris negatif = {flag[neighbor_index[~flag]].mean():.3f}")

Pasangan menurut jumlah fitur berbeda:

 {0: 14, 1: 635, 2: 587, 3: 1479}

===== drop_300 (porsi positif 0.058) =====


Logistic Regression: 0.5359 (0.4386-0.6218)


RandomForest daun=1: 0.2912 (0.2206-0.3972)


RandomForest daun=5: 0.3445 (0.2714-0.4164)


RandomForest daun=20: 0.4228 (0.3497-0.4977)


... (6 outputs left out)

RandomForest daun=20: 0.3824 (0.3239-0.4191)


RandomForest, grup hampir kembar (beda <= 1 fitur): 0.2449 (0.1940-0.2895)


RandomForest, grup hampir kembar (beda <= 2 fitur): 0.2225 (0.1534-0.2630)


RandomForest, grup hampir kembar (beda <= 3 fitur): 0.2198 (0.1437-0.3343)
Porsi positif di 10 tetangga: baris positif = 0.011 | baris negatif = 0.083


## Struktur posisi

Pertanyaan: apakah kemiripan menurut posisi baris lebih luas daripada duplikat persis?
Pasangan yang identik pada semua fitur dikeluarkan, supaya yang diuji adalah struktur **di luar** duplikat persis.
`Age Group` tidak dipakai di sini karena informasinya sama dengan `Age`, dan `Churn` tidak dipakai agar uji ini hanya melihat fitur.

### P1. Pemindaian semua jarak

Statistik tiap jarak: rata-rata jumlah fitur yang sama pada pasangan baris berjarak itu, tanpa ambang buatan.
Urutan baris diacak berulang untuk membentuk sebaran null, lalu nilai z tertinggi dari tiap permutasi dipakai sebagai batas agar banyaknya jarak yang diuji sudah diperhitungkan.

In [12]:
POSITION_FEATURES = [col for col in FEATURES if col != "Age Group"]
MAX_LAG = 3000
N_POSITION_PERMUTATIONS = 200
position_values = df[POSITION_FEATURES].to_numpy(dtype=float)
n_position_features = position_values.shape[1]


def lag_profile(values):
    # Rata-rata jumlah fitur yang sama per jarak, tanpa pasangan yang identik pada semua fitur
    profile = np.empty(MAX_LAG)
    for lag in range(1, MAX_LAG + 1):
        matches = (values[:-lag] == values[lag:]).sum(axis=1)
        profile[lag - 1] = matches[matches < n_position_features].mean()
    return profile


observed_profile = lag_profile(position_values)
rng_position = np.random.default_rng(SEED)
null_profiles = np.array([
    lag_profile(position_values[rng_position.permutation(len(position_values))])
    for _ in range(N_POSITION_PERMUTATIONS)
])
null_mean, null_sd = null_profiles.mean(axis=0), null_profiles.std(axis=0, ddof=1)
observed_z = (observed_profile - null_mean) / null_sd
max_null_z = ((null_profiles - null_mean) / null_sd).max(axis=1)
z_threshold = np.quantile(max_null_z, 0.95)

lags = np.arange(1, MAX_LAG + 1)
is_multiple_50 = lags % 50 == 0
significant = observed_z > z_threshold
print(f"Batas z (kuantil 95% dari z maksimum null): {z_threshold:.2f}")
print(f"Jarak yang melewati batas: {significant.sum()} dari {MAX_LAG}")
print(f"Di antaranya kelipatan 50: {(significant & is_multiple_50).sum()} dari {is_multiple_50.sum()} kelipatan 50")
print(f"Bukan kelipatan 50: {(significant & ~is_multiple_50).sum()} dari {(~is_multiple_50).sum()}")
print(f"Median z: kelipatan 50 = {np.median(observed_z[is_multiple_50]):.2f} | lainnya = {np.median(observed_z[~is_multiple_50]):.2f}")
print(f"Rata-rata fitur sama: kelipatan 50 = {observed_profile[is_multiple_50].mean():.3f} | "
      f"lainnya = {observed_profile[~is_multiple_50].mean():.3f} | null = {null_mean.mean():.3f}")
top = np.argsort(observed_z)[::-1][:15]
print()
print(pd.DataFrame({"jarak": lags[top], "z": observed_z[top].round(1), "fitur_sama": observed_profile[top].round(3)}).to_string(index=False))

Batas z (kuantil 95% dari z maksimum null): 4.03
Jarak yang melewati batas: 464 dari 3000
Di antaranya kelipatan 50: 60 dari 60 kelipatan 50
Bukan kelipatan 50: 404 dari 2940
Median z: kelipatan 50 = 43.86 | lainnya = -2.85
Rata-rata fitur sama: kelipatan 50 = 4.399 | lainnya = 3.174 | null = 3.254

 jarak     z  fitur_sama
   300 101.7       5.118
   200  94.7       4.947
    50  89.4       4.705
   400  88.9       4.821
   100  87.6       4.833
   500  80.3       4.628
   250  79.0       4.626
   150  77.4       4.602
   650  75.0       4.591
   600  70.9       4.600
   350  68.6       4.535
   450  68.1       4.539
   950  65.5       4.542
   550  65.4       4.548
   850  63.3       4.498


In [13]:
# Kolom mana yang lebih sering sama pada jarak kelipatan 50?
match_total = {True: np.zeros(n_position_features), False: np.zeros(n_position_features)}
pair_total = {True: 0, False: 0}
for lag in lags:
    same = position_values[:-lag] == position_values[lag:]
    same = same[same.sum(axis=1) < n_position_features]
    key = bool(lag % 50 == 0)
    match_total[key] += same.sum(axis=0)
    pair_total[key] += len(same)

column_match = pd.DataFrame({
    "kelipatan_50": match_total[True] / pair_total[True],
    "lainnya": match_total[False] / pair_total[False],
}, index=POSITION_FEATURES)
column_match["selisih"] = column_match["kelipatan_50"] - column_match["lainnya"]
print(column_match.round(4).sort_values("selisih", ascending=False).to_string())

                         kelipatan_50  lainnya  selisih
Age                            0.7084   0.3253   0.3831
Status                         0.9748   0.6196   0.3552
Charge  Amount                 0.4918   0.3714   0.1204
Tariff Plan                    0.9709   0.8543   0.1166
Complains                      0.9060   0.8578   0.0482
Distinct Called Numbers        0.0668   0.0189   0.0479
Call  Failure                  0.1216   0.0796   0.0420
Frequency of use               0.0432   0.0076   0.0356
Frequency of SMS               0.0767   0.0446   0.0321
Seconds of Use                 0.0292   0.0024   0.0268
Subscription  Length           0.0724   0.0466   0.0259
Customer Value                 0.0123   0.0015   0.0107


### P2. Salinan yang hampir sama atau hanya siklus?

Untuk tiap baris unik, cari satu tetangga terdekat lalu lihat jaraknya di dalam file.
Jika posisi tidak berkaitan dengan kemiripan, porsi kelipatan 50 seharusnya mendekati porsi pada urutan yang diacak.

In [14]:
position_unique = df.drop_duplicates(subset=POSITION_FEATURES)
position_index = position_unique.index.to_numpy()
scaled_position = StandardScaler().fit_transform(position_unique[POSITION_FEATURES])
distances, indices = NearestNeighbors(n_neighbors=2).fit(scaled_position).kneighbors(scaled_position)
nn_distance_unique, nn_index_unique = distances[:, 1], indices[:, 1]
nn_lag = np.abs(position_index - position_index[nn_index_unique])

rng_nn = np.random.default_rng(SEED)
null_share = np.empty(N_PERMUTATIONS)
for i in range(N_PERMUTATIONS):
    shuffled = rng_nn.permutation(position_index)
    null_share[i] = (np.abs(shuffled - shuffled[nn_index_unique]) % 50 == 0).mean()
observed_nn_share = (nn_lag % 50 == 0).mean()
print(f"Baris unik: {len(position_unique):,}")
print(f"Porsi tetangga terdekat pada jarak kelipatan 50: {observed_nn_share:.3f} | null rata-rata = {null_share.mean():.3f}, "
      f"null maks = {null_share.max():.3f} | p = {(1 + (null_share >= observed_nn_share).sum()) / (1 + N_PERMUTATIONS):.4f}")

bands = pd.qcut(nn_distance_unique, q=4)
print()
print(pd.DataFrame({"kuartil_jarak": bands, "kelipatan_50": nn_lag % 50 == 0}).groupby("kuartil_jarak", observed=True)["kelipatan_50"]
      .agg(["size", "mean"]).round(3).rename(columns={"size": "baris", "mean": "porsi_kelipatan_50"}).to_string())
print()
print("Jarak tetangga terdekat paling sering:")
print(pd.Series(nn_lag).value_counts().head(10).rename("jumlah_baris").to_string())

Baris unik: 2,836
Porsi tetangga terdekat pada jarak kelipatan 50: 0.886 | null rata-rata = 0.020, null maks = 0.030 | p = 0.0005

                   baris  porsi_kelipatan_50
kuartil_jarak                               
(-0.00076, 0.216]    710               0.883
(0.216, 0.282]       709               0.951
(0.282, 0.383]       708               0.893
(0.383, 3.723]       709               0.819

Jarak tetangga terdekat paling sering:
950     245
650     239
200     186
400     174
100     160
550     152
300     146
2200    122
1250     83
1150     82


### P3. Apakah nilai kolom mengikuti posisi?

`eta2` adalah porsi variasi kolom yang dijelaskan oleh kelompok posisi. Nilainya dibandingkan dengan urutan baris yang diacak.
Kelompok posisi yang diuji: sisa bagi 50 (siklus), blok 50 baris, dan blok 1.050 baris.

In [15]:
def eta_squared(values, labels):
    total = ((values - values.mean()) ** 2).sum()
    counts = np.bincount(labels)
    means = np.bincount(labels, weights=values) / counts
    return (counts * (means - values.mean()) ** 2).sum() / total


positions = np.arange(len(df))
position_groups = {"sisa_bagi_50": positions % 50, "blok_50": positions // 50, "blok_1050": positions // 1050}
rng_eta = np.random.default_rng(SEED)
rows = []
for col in ALL_COLUMNS:
    values = df[col].to_numpy(dtype=float)
    row = {"kolom": col}
    for label, groups in position_groups.items():
        observed_eta = eta_squared(values, groups)
        null_eta = np.array([eta_squared(rng_eta.permutation(values), groups) for _ in range(N_PERMUTATIONS)])
        row[label] = observed_eta
        row[f"{label}_null95"] = np.quantile(null_eta, 0.95)
    rows.append(row)
print(pd.DataFrame(rows).round(4).to_string(index=False))
print()
print(df.groupby(positions // 1050)[["Subscription  Length", "Seconds of Use", "Customer Value", "Status", TARGET]].mean().round(3).to_string())

                  kolom  sisa_bagi_50  sisa_bagi_50_null95  blok_50  blok_50_null95  blok_1050  blok_1050_null95
          Call  Failure        0.6300               0.0208   0.3434          0.0260     0.2270            0.0019
              Complains        0.3481               0.0214   0.0148          0.0256     0.0021            0.0019
   Subscription  Length        0.8448               0.0213   0.1551          0.0256     0.0362            0.0019
         Charge  Amount        0.6872               0.0208   0.2059          0.0263     0.0876            0.0019
         Seconds of Use        0.9909               0.0209   0.0087          0.0257     0.0034            0.0018
       Frequency of use        0.9794               0.0207   0.0199          0.0259     0.0070            0.0019
       Frequency of SMS        0.9857               0.0208   0.0128          0.0258     0.0019            0.0019
Distinct Called Numbers        0.8973               0.0214   0.0949          0.0258     0.0086  

## Verifikasi struktur 50 kelompok

Baris disusun menjadi matriks 63 blok x 50 kelompok, dengan kelompok = sisa bagi 50 dari posisi baris dan blok = hasil bagi 50.
Jika semua kelompok digeser dengan pola yang sama, selisih terhadap blok pertama pada satu blok akan sama di hampir semua kelompok.

### V1. Contoh baris mentah

In [16]:
GROUP_SIZE, N_BLOCKS = 50, 63
assert GROUP_SIZE * N_BLOCKS == len(df)
row_group = np.arange(len(df)) % GROUP_SIZE
row_block = np.arange(len(df)) // GROUP_SIZE
print(df.iloc[[0, 50, 100, 150]].to_string())
print()
print(df.iloc[[1, 51, 101, 151]].to_string())

     Call  Failure  Complains  Subscription  Length  Charge  Amount  Seconds of Use  Frequency of use  Frequency of SMS  Distinct Called Numbers  Age Group  Tariff Plan  Status  Age  Customer Value  Churn
0                8          0                    38               0            4370                71                 5                       17          3            1       1   30          197.64      0
50              11          0                    40               1            4430                74                10                       19          3            1       1   30          220.16      0
100              5          0                    36               0            4310                68                 0                       15          2            1       1   25          197.01      0
150              9          0                    42               1            4450                73                15                       21          3            1       1   3

### V2. Uji pola selisih pada semua kelompok

Untuk tiap blok setelah blok pertama, cari selisih yang paling sering muncul di 50 kelompok, lalu hitung porsi kelompok yang mengikutinya.
Pembanding: urutan kelompok diacak di dalam tiap blok, sehingga kelompok tidak lagi sejajar.

In [17]:
def offset_agreement(matrix):
    offsets = matrix[1:] - matrix[0]
    shares, modes = [], []
    for block_offsets in offsets:
        values, counts = np.unique(block_offsets, return_counts=True)
        shares.append(counts.max() / len(block_offsets))
        modes.append(values[counts.argmax()])
    return np.array(shares), np.array(modes)


OFFSET_COLUMNS = ["Seconds of Use", "Frequency of use", "Frequency of SMS", "Distinct Called Numbers",
                  "Subscription  Length", "Call  Failure", "Charge  Amount", "Customer Value",
                  "Complains", "Tariff Plan", "Status", "Age", TARGET]
rng_offset = np.random.default_rng(SEED)
block_modes, rows = {}, []
for col in OFFSET_COLUMNS:
    matrix = df[col].to_numpy(dtype=float).reshape(N_BLOCKS, GROUP_SIZE)
    shares, modes = offset_agreement(matrix)
    block_modes[col] = modes
    null_means = [offset_agreement(rng_offset.permuted(matrix, axis=1))[0].mean() for _ in range(N_PERMUTATIONS // 10)]
    is_modal = (matrix[1:] - matrix[0]) == modes[:, None]
    deviates = ~is_modal
    at_minimum = matrix[1:] == df[col].min()
    at_maximum = matrix[1:] == df[col].max()
    rows.append({
        "kolom": col,
        "porsi_ikut_pola": shares.mean(),
        "null_rata2": np.mean(null_means),
        "blok_min": shares.min(),
        "kelompok_100pct": is_modal.all(axis=0).sum(),
        "menyimpang": deviates.sum(),
        "menyimpang_di_min_maks": (deviates & (at_minimum | at_maximum)).sum(),
        "selisih_unik": len(np.unique(modes)),
    })
print(pd.DataFrame(rows).round(3).to_string(index=False))

                  kolom  porsi_ikut_pola  null_rata2  blok_min  kelompok_100pct  menyimpang  menyimpang_di_min_maks  selisih_unik
         Seconds of Use            0.911       0.028      0.72               32         277                     148            47
       Frequency of use            0.944       0.047      0.72               36         173                     139            31
       Frequency of SMS            0.775       0.072      0.30               15         697                     553            45
Distinct Called Numbers            0.934       0.067      0.56               28         206                     133            23
   Subscription  Length            0.999       0.155      0.96               48           4                       4            17
          Call  Failure            0.790       0.113      0.18                8         650                     575            23
         Charge  Amount            0.762       0.445      0.60                0         73

### V3. Pergeseran antarblok 1.050 baris dan `Churn` per kelompok

In [18]:
subscription = df["Subscription  Length"].to_numpy()
print("Blok kedua dikurangi blok pertama:", pd.Series(subscription[1050:2100] - subscription[:1050]).value_counts().to_dict())
print("Blok ketiga dikurangi blok pertama:", pd.Series(subscription[2100:3150] - subscription[:1050]).value_counts().to_dict())

group_churn = df.groupby(row_group)[TARGET].sum()
print("\nJumlah baris churn per kelompok (dari 63 baris), dengan jumlah kelompoknya:")
print(group_churn.value_counts().sort_index().to_dict())
churn_rate_by_group = group_churn / N_BLOCKS
print(f"Churn <= 5%: {(churn_rate_by_group <= 0.05).sum()} kelompok | >= 95%: {(churn_rate_by_group >= 0.95).sum()} kelompok | "
      f"di antaranya: {((churn_rate_by_group > 0.05) & (churn_rate_by_group < 0.95)).sum()} kelompok")
print(f"Baris churn pada kelompok >= 95%: {group_churn[churn_rate_by_group >= 0.95].sum()} dari {group_churn.sum()}")

Blok kedua dikurangi blok pertama: {-2: 1046, 0: 2, -1: 2}
Blok ketiga dikurangi blok pertama: {2: 1050}

Jumlah baris churn per kelompok (dari 63 baris), dengan jumlah kelompoknya:
{0: 35, 1: 3, 2: 1, 3: 2, 5: 1, 38: 1, 63: 7}
Churn <= 5%: 41 kelompok | >= 95%: 7 kelompok | di antaranya: 2 kelompok
Baris churn pada kelompok >= 95%: 441 dari 495


### V4. Apakah duplikat adalah efek samping struktur ini?

Jika iya, pasangan kembar harus berada pada kelompok yang sama dan berasal dari blok dengan pola selisih yang sama.

In [19]:
pair_group_same = (pairs[:, 0] % GROUP_SIZE) == (pairs[:, 1] % GROUP_SIZE)
print(f"Pasangan kembar di kelompok yang sama: {pair_group_same.sum()} dari {len(pairs)}")
print(f"Beraktivitas: {(pair_group_same & pair_is_active).sum()} dari {pair_is_active.sum()} | "
      f"beraktivitas nol: {(pair_group_same & ~pair_is_active).sum()} dari {(~pair_is_active).sum()}")

shift_columns = ["Seconds of Use", "Frequency of use", "Frequency of SMS", "Distinct Called Numbers", "Subscription  Length"]
mode_vectors = np.column_stack([np.r_[0, block_modes[col]] for col in shift_columns])
same_vector = (mode_vectors[pairs[:, 0] // GROUP_SIZE] == mode_vectors[pairs[:, 1] // GROUP_SIZE]).all(axis=1)
print(f"\nPola selisih utama: {len(np.unique(mode_vectors, axis=0))} pola unik dari {N_BLOCKS} blok")
print(f"Pasangan kembar dari blok dengan pola selisih sama: {same_vector.sum()} dari {len(pairs)} | "
      f"beraktivitas: {(same_vector & pair_is_active).sum()} dari {pair_is_active.sum()}")
block_pairs = pd.Series(list(zip(pairs[:, 0] // GROUP_SIZE, pairs[:, 1] // GROUP_SIZE))).value_counts()
print(f"Pasangan blok yang menghasilkan kembar: {len(block_pairs)} | teratas: "
      f"{ {tuple(map(int, key)): int(value) for key, value in block_pairs.head(5).items()} }")

Pasangan kembar di kelompok yang sama: 426 dari 457
Beraktivitas: 365 dari 365 | beraktivitas nol: 61 dari 92

Pola selisih utama: 57 pola unik dari 63 blok
Pasangan kembar dari blok dengan pola selisih sama: 373 dari 457 | beraktivitas: 362 dari 365
Pasangan blok yang menghasilkan kembar: 66 | teratas: {(61, 62): 46, (40, 41): 44, (19, 20): 43, (21, 41): 42, (42, 62): 42}


### V5. Prediksi `Churn`: CV acak vs grup per kelompok

Grup per kelompok memastikan profil yang sama tidak muncul di train dan test sekaligus.
Fold yang test set-nya hanya berisi satu kelas dilewati dan jumlahnya dilaporkan.

In [20]:
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

CV_MODELS = {
    "LogisticRegression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
}


def evaluate(make_model, features, labels, splits):
    roc, pr, skipped = [], [], 0
    for train_index, test_index in splits:
        if len(np.unique(labels[test_index])) < 2:
            skipped += 1
            continue
        score = make_model().fit(features.iloc[train_index], labels[train_index]).predict_proba(features.iloc[test_index])[:, 1]
        roc.append(roc_auc_score(labels[test_index], score))
        pr.append(average_precision_score(labels[test_index], score))
    return {"roc_auc": np.mean(roc), "roc_min": np.min(roc), "roc_maks": np.max(roc), "pr_auc": np.mean(pr),
            "pr_min": np.min(pr), "pr_maks": np.max(pr), "fold_dipakai": len(roc), "fold_dilewati": skipped}


def repeated_splits(features, labels, groups=None, n_repeats=5):
    splits = []
    for repeat in range(n_repeats):
        if groups is None:
            splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + repeat)
            splits += list(splitter.split(features, labels))
        else:
            splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + repeat)
            splits += list(splitter.split(features, labels, groups))
    return splits


labels = df[TARGET].to_numpy()
results = {}
for model_name, make_model in CV_MODELS.items():
    results[(model_name, "acak")] = evaluate(make_model, df[FEATURES], labels, repeated_splits(df[FEATURES], labels))
    group_splits = repeated_splits(df[FEATURES], labels, row_group)
    assert all(not set(row_group[a]) & set(row_group[b]) for a, b in group_splits)
    results[(model_name, "grup_kelompok")] = evaluate(make_model, df[FEATURES], labels, group_splits)
print(pd.DataFrame(results).T.round(4).to_string())
print(f"\nBaseline PR-AUC (churn rate): {labels.mean():.4f}")

                                  roc_auc  roc_min  roc_maks  pr_auc  pr_min  pr_maks  fold_dipakai  fold_dilewati
LogisticRegression acak            0.9324   0.9133    0.9586  0.7552  0.6899   0.8470          25.0            0.0
                   grup_kelompok   0.8408   0.6072    1.0000  0.5712  0.1624   1.0000          25.0            0.0
RandomForest       acak            0.9844   0.9755    0.9910  0.9299  0.8970   0.9564          25.0            0.0
                   grup_kelompok   0.7904   0.3694    0.9984  0.4537  0.0352   0.8215          25.0            0.0

Baseline PR-AUC (churn rate): 0.1571


### V6. Classifier two-sample test dengan grup per kelompok

Mengulang B3, tetapi profil yang sama tidak muncul di train dan test sekaligus.

In [21]:
unique_group = np.where(is_unique_row)[0] % GROUP_SIZE
forest_factory = lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1)
two_sample = {}
for name, flag in definitions.items():
    flag_int = flag.astype(int)
    splits = repeated_splits(unique_rows[FEATURES], flag_int, unique_group)
    two_sample[name] = evaluate(forest_factory, unique_rows[FEATURES], flag_int, splits)
print(pd.DataFrame(two_sample).T[["roc_auc", "roc_min", "roc_maks", "fold_dipakai", "fold_dilewati"]].round(4).to_string())

          roc_auc  roc_min  roc_maks  fold_dipakai  fold_dilewati
drop_300   0.6583   0.5986    0.7408          25.0            0.0
drop_253   0.6459   0.5496    0.7130          25.0            0.0
